In [1]:
from dotenv import load_dotenv
from anthropic import Anthropic
from sentence_transformers import SentenceTransformer
import numpy as np

load_dotenv(override=True)
client = Anthropic()
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

/Users/rishithreddy/dummy/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13125.92it/s]


In [2]:
# --- knowledge base (same as before) ---
docs = [
    "The capital of France is Paris.",
    "Python is a popular programming language for data science.",
    "The Eiffel Tower is located in Paris, France.",
    "Photosynthesis is how plants convert sunlight into energy.",
    "Machine learning models learn patterns from data.",
]
doc_embeddings = model.encode(docs, normalize_embeddings=True)

In [3]:
# --- RETRIEVAL: find top-k chunks for a query ---
def retrieve(query, k=2):
    q = model.encode(query, normalize_embeddings=True)
    scores = doc_embeddings @ q
    top = np.argsort(scores)[::-1][:k]     # best k positions
    return [docs[i] for i in top]   

In [4]:
# --- GENERATION: answer using only the retrieved chunks ---
def rag_answer(query):
    chunks = retrieve(query)
    context = "\n".join(f"- {c}" for c in chunks)   # format chunks for the prompt

    resp = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=256,
        system="Answer the question using ONLY the context provided. "
               "If the answer isn't in the context, say you don't know.",
        messages=[{
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {query}"
        }],
    )
    answer = "".join(b.text for b in resp.content if b.type == "text")
    return answer, chunks

In [15]:
# --- try it ---
query = "where is paris located?"
answer, used_chunks = rag_answer(query)
print(f"Answer: {answer}")
print(f"Used chunks: {used_chunks}")

Answer: Based on the context provided, Paris is located in France.
Used chunks: ['The capital of France is Paris.', 'The Eiffel Tower is located in Paris, France.']


In [16]:
print("\nRetrieved chunks used:")
for c in used_chunks:
    print(" -", c)


Retrieved chunks used:
 - The capital of France is Paris.
 - The Eiffel Tower is located in Paris, France.
